# Jamo — 클래스에서 자모를 골라 학습 데이터 생성
GPU 런타임에서 위부터 실행하세요. unique_images는 자모 원본 은행입니다. 같은 클래스의 초성·중성·종성을 각각 골라 합성한 별도의 학습 데이터를 만듭니다.

N은 0~5 클래스, M은 0 초성/1 중성/2 종성입니다. 파일명의 글자는 선택한 M의 자모 종류를 알려줍니다. 예: 가의 ㄱ은 0_0_가.png에서, ㅏ는 0_1_아.png에서 가져올 수 있습니다. 클래스 0~2는 두 자모, 3~5는 세 자모를 합칩니다.

원본은 한 번씩 병렬로 읽고 합성과 미리보기는 Colab 로컬에서 실행합니다. Drive에는 학습 데이터 ZIP과 학습 결과를 저장합니다.
[데이터와 실행 안내](https://github.com/yechan25/hsqm/blob/main/docs/stroke_model/JAMO.md)


In [ ]:
from pathlib import Path
import sys, subprocess, tempfile, json
root = Path(tempfile.mkdtemp(prefix="jamo_", dir="/content"))
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/yechan25/hsqm.git", str(root)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(root / "requirements.txt")], check=True)
sys.path.insert(0, str(root))
for key in list(sys.modules):
    if key == "stroke_model" or key.startswith("stroke_model."):
        del sys.modules[key]
commit = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip()
print("Jamo 코드 준비 완료 / GitHub 버전:", commit)


## 데이터와 학습 설정
10개 폰트 모두 샘플마다 reference와 input의 폰트가 같습니다. 각 폰트의 원본 자모를 reference로 쓰고 같은 자모를 변형하여 input과 정답을 만듭니다. 초성·중성·종성은 항상 이 순서이며 기준은 원본, 타깃은 자모별 scale/rotation/shift를 적용합니다.

타깃 폰트는 생성 전에 train/val로 나눕니다. 학습/검증 분리를 위해 최소 2개 폰트가 필요합니다. SAMPLES_PER_CLASS는 폰트·클래스별 최대 조합 수이며 먼저 100개로 실행합니다. 원본 이미지를 수정했거나 추가했다면 REBUILD=True로 캐시를 새로 만드세요.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SOURCE_ROOT = "/content/drive/Shareddrives/2026 자율연구/jamo seperation/unique_images"
REFERENCE_FONT = None  # 모든 샘플에서 자신의 폰트를 reference로 사용
OUTPUT_ROOT = "/content/drive/Shareddrives/2026 자율연구/jamo seperation/jamo_training"
LOCAL_DATA_ROOT = "/content/jamo_prepared"
SAMPLES_PER_CLASS = 100  # 빠른 확인은 10; 폰트·클래스별 최대 생성 조합 수
SOURCE_WORKERS = 8  # 원본 PNG 병렬 읽기
MAX_OVERLAP = 0.001  # 두 자모 이상 겹친 픽셀 / 전체 캔버스
REBUILD = False  # 원본 변경 시 True; 재실행은 ZIP 또는 로컬 캐시 사용
PREVIEW_COUNT = 3
VARIANTS = 0  # 합성 때 이미 자모별 변형을 적용함. 추가 전체 글자 증강은 선택
EPOCHS = 30
BATCH_SIZE = 8
SEED = 42
TEST_CSV = ""  # 선택: 별도 held-out test CSV


In [ ]:
from stroke_model.prepare_jamo import prepare_jamo
output_root = Path(OUTPUT_ROOT)
manifest = prepare_jamo(
    SOURCE_ROOT, output_root / "data", reference_font=REFERENCE_FONT,
    seed=SEED, rebuild=REBUILD, samples_per_class=SAMPLES_PER_CLASS,
    workers=SOURCE_WORKERS, local_root=LOCAL_DATA_ROOT, max_overlap=MAX_OVERLAP,
)
report = json.loads((manifest.parent / "conversion_report.json").read_text(encoding="utf-8"))
print("기준 방식:", report["reference_mode"], "(reference/input 동일 폰트)")
print("검증 폰트:", report["validation_fonts"])
print("생성한 샘플:", len(report["rows"]))
print("제외된 조합/클래스:", len(report["skipped"]))
print("제외 예시:", report["skipped"][:5])
print("학습 데이터 ZIP:", output_root / "data")
print("로컬 manifest:", manifest)


## 생성 데이터 확인
로컬 데이터 중 설정한 개수만 표시합니다. 이 셀을 다시 실행해도 합성을 반복하지 않습니다.


In [ ]:
import matplotlib.pyplot as plt
from stroke_model.preview_reference import preview_reference
from stroke_model.train_reference import read_manifest, StrokeDataset
rows = read_manifest(manifest)
if any(r.get("reference_font") != r.get("font") for r in rows):
    raise ValueError("이전 교차 폰트 데이터입니다. 최신 데이터 생성 셀을 먼저 실행하세요.")
dataset = StrokeDataset(rows, size=128)
for index in range(min(PREVIEW_COUNT, len(dataset))):
    image, reference, strokes, labels = dataset[index]
    row = rows[index]
    print(row["font"], "class", row["class_id"], row["char_id"], row.get("target_components", []))
    fig, axes = plt.subplots(2, len(strokes) + 1, squeeze=False,
                             figsize=(2.5 * (len(strokes) + 1), 5))
    axes[0, 0].imshow(reference[0], cmap="gray_r", vmin=0, vmax=1)
    axes[0, 0].set_title("Reference composite")
    axes[1, 0].imshow(image[0], cmap="gray_r", vmin=0, vmax=1)
    axes[1, 0].set_title("Transformed input")
    for k in range(len(strokes)):
        axes[0, k+1].imshow(strokes[k], cmap="gray_r", vmin=0, vmax=1)
        axes[0, k+1].set_title(f"Reference jamo {k+1}")
        axes[1, k+1].imshow(labels == k, cmap="gray_r", vmin=0, vmax=1)
        axes[1, k+1].set_title(f"Target jamo {k+1}")
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
from stroke_model.reference_cache import ensure_augmented
training_manifest = ensure_augmented(manifest, variants=VARIANTS, seed=SEED)
print("학습 manifest:", training_manifest)


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Colab 메뉴: 런타임 → 런타임 유형 변경 → GPU를 선택하세요.")
output = output_root / ("jamo_run_" + datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S_%f"))
output.mkdir(parents=True, exist_ok=False)
settings = dict(experiment="jamo", code_commit=commit, source_root=SOURCE_ROOT, reference_mode=report["reference_mode"],
                manifest=str(manifest), training_manifest=str(training_manifest),
                variants=VARIANTS, samples_per_class=SAMPLES_PER_CLASS, max_overlap=MAX_OVERLAP,
                epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED)
(output / "run_config.json").write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf-8")
cmd = [sys.executable, "-u", "-m", "stroke_model.train_reference",
       "--manifest", str(training_manifest), "--epochs", str(EPOCHS),
       "--batch-size", str(BATCH_SIZE), "--num-workers", "2", "--seed", str(SEED),
       "--output", str(output)]
print("이미 생성된 로컬 데이터로 학습합니다. 체크포인트 저장:", output, flush=True)
with subprocess.Popen(cmd, cwd=root, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        result = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
        raise
if result:
    raise RuntimeError(f"학습 실패: 종료 코드 {result}. 위 로그를 확인하세요.")
print("체크포인트:", output / "best.pt")


## 학습 결과
검증 Dice는 정답이 지정된 픽셀에 대한 점수입니다. 회색으로 표시되는 미할당 영역은 평가에서 제외됩니다. 자모로 묶으면 과제가 달라지므로 기존 개별 획 Dice와 직접 비교하지 마세요.


In [ ]:
import pandas as pd
history = pd.read_json(output / "history.json")
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history["epoch"], history["train_loss"])
axes[0].set(title="Training loss", xlabel="Epoch")
axes[1].plot(history["epoch"], history["val_macro_dice"])
axes[1].set(title="Validation Dice (annotated only)", xlabel="Epoch", ylim=(0, 1))
plt.tight_layout()
fig.savefig(output / "learning_curves.png", dpi=150)
plt.show()
plt.close(fig)
display(history.loc[[history["val_macro_dice"].idxmax()]])
preview_reference(manifest, checkpoint=output / "best.pt", count=5,
                  save_dir=output / "val_preview", show_soft=True)
print("학습 곡선, 비교 이미지, soft mask 저장:", output)


## 선택: 별도 test 데이터 평가
TEST_CSV가 비어 있으면 건너뜁니다. 학습/검증 CSV를 test로 다시 지정하지 마세요. 동일 경로의 test 이미지를 수정했다면 새 CSV 파일명으로 저장하여 기존 test 캐시와 구분하세요.


In [ ]:
if TEST_CSV.strip():
    from stroke_model.test_reference import run_test
    test_csv = existing_path(TEST_CSV)
    test_metrics = run_test(output / "best.pt", test_csv, preview_count=5)
else:
    print("별도 test CSV 미지정: 위 결과는 검증 세트 결과입니다.")
